# UC-02: Invoice Three-Way Match — Training Pipeline

Models: Logistic Regression → Random Forest → XGBoost (Optuna) → LightGBM (Optuna)

**Setup:**
- CV: RepeatedStratifiedKFold(5, 3) → 15 evaluations
- Primary metric: F1 (binary)
- MLflow tracking: local file store
- Class imbalance: class_weight/scale_pos_weight (NO SMOTE — 320 rows too few)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style="whitegrid")
optuna.logging.set_verbosity(optuna.logging.WARNING)

from ml.common.db_config import load_tables
from ml.data_processing.python.uc02_preprocessing import UC02_TABLES
from ml.uc_02_invoice_match.feature_engineering.feature_functions import (
    build_uc02_features, prepare_feature_matrix,
)

In [ ]:
# Load data and build features
csv_dir = project_root / "output" / "csv"
tables = load_tables("csv", UC02_TABLES, csv_dir=csv_dir)

feature_df = build_uc02_features(tables, leave_one_out=True)
X, y = prepare_feature_matrix(feature_df, target="binary")

print(f"Features: {X.shape}, Target: {dict(y.value_counts())}")

# MLflow setup
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("uc02_invoice_match")

## 2. Cross-Validation Strategy

With only 320 rows and no temporal ordering guarantees, we use:
- **RepeatedStratifiedKFold(n_splits=5, n_repeats=3)**
- 15 total evaluations for robust estimates
- Stratified to preserve class proportions in each fold

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Verify fold sizes
for i, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    if i < 5:  # Show first repeat only
        train_pos = y.iloc[train_idx].sum()
        test_pos = y.iloc[test_idx].sum()
        print(f"Fold {i+1}: train={len(train_idx)} (pos={train_pos}), test={len(test_idx)} (pos={test_pos})")

## 3. Metrics

- **Primary:** F1 (harmonic mean of precision and recall)
- **Secondary:** Precision, Recall, AUC-ROC, Average Precision

In [ ]:
SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
}

def evaluate_model(model, X, y, cv, name):
    """Run cross-validation and log to MLflow."""
    scores = cross_validate(model, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    results = {k: (scores[f"test_{k}"].mean(), scores[f"test_{k}"].std()) for k in SCORING}

    with mlflow.start_run(run_name=name):
        for metric, (mean, std) in results.items():
            mlflow.log_metric(f"cv_{metric}_mean", mean)
            mlflow.log_metric(f"cv_{metric}_std", std)

    print(f"\n{name}:")
    for metric, (mean, std) in results.items():
        print(f"  {metric:15s}: {mean:.4f} (+/- {std:.4f})")

    return results

model_results = {}  # Store all results for comparison

## 4. Model 1: Logistic Regression (Baseline)

In [ ]:
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=42)),
])

model_results["Logistic Regression"] = evaluate_model(lr_model, X, y, cv, "logistic_regression")

## 5. Model 2: Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42,
)

model_results["Random Forest"] = evaluate_model(rf_model, X, y, cv, "random_forest")

## 6. Model 3: XGBoost + Optuna

50 Optuna trials with the following search space:
- n_estimators: 50-300
- max_depth: 2-8
- learning_rate: 0.01-0.3 (log scale)
- subsample: 0.6-1.0
- colsample_bytree: 0.6-1.0
- min_child_weight: 1-10
- reg_alpha/lambda: 1e-8 to 10.0 (log scale)
- scale_pos_weight: 245/75 (class ratio)

In [ ]:
n_pos = int(y.sum())
n_neg = len(y) - n_pos
scale_pos = n_neg / max(n_pos, 1)
print(f"Class ratio — neg:pos = {n_neg}:{n_pos}, scale_pos_weight = {scale_pos:.2f}")

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": scale_pos,
        "eval_metric": "logloss",
        "random_state": 42,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction="maximize", study_name="xgboost_uc02")
xgb_study.optimize(xgb_objective, n_trials=50, timeout=600)

print(f"\nBest XGBoost F1: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")

In [ ]:
# Evaluate best XGBoost
best_xgb_params = xgb_study.best_params.copy()
best_xgb_params["scale_pos_weight"] = scale_pos
best_xgb_params["eval_metric"] = "logloss"
best_xgb_params["random_state"] = 42

xgb_model = xgb.XGBClassifier(**best_xgb_params)
model_results["XGBoost"] = evaluate_model(xgb_model, X, y, cv, "xgboost_optuna")

## 7. Model 4: LightGBM + Optuna

In [ ]:
def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "is_unbalance": True,
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)
    return scores.mean()

lgb_study = optuna.create_study(direction="maximize", study_name="lightgbm_uc02")
lgb_study.optimize(lgb_objective, n_trials=50, timeout=600)

print(f"\nBest LightGBM F1: {lgb_study.best_value:.4f}")
print(f"Best params: {lgb_study.best_params}")

In [ ]:
best_lgb_params = lgb_study.best_params.copy()
best_lgb_params["is_unbalance"] = True
best_lgb_params["random_state"] = 42
best_lgb_params["verbose"] = -1

lgb_model = lgb.LGBMClassifier(**best_lgb_params)
model_results["LightGBM"] = evaluate_model(lgb_model, X, y, cv, "lightgbm_optuna")

## 8. Multi-class Variant

In [ ]:
# Retrain best model with multi-class target
X_mc, y_mc = prepare_feature_matrix(feature_df, target="multiclass")

mc_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    **{k: v for k, v in best_xgb_params.items()
       if k not in ["scale_pos_weight", "eval_metric"]},
    eval_metric="mlogloss",
)

mc_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
mc_scores = cross_val_score(mc_model, X_mc, y_mc, cv=mc_cv, scoring="f1_weighted", n_jobs=-1)
print(f"Multi-class XGBoost F1 (weighted): {mc_scores.mean():.4f} (+/- {mc_scores.std():.4f})")

## 9. Model Comparison

In [ ]:
# Summary table
comparison = pd.DataFrame({
    name: {metric: f"{mean:.4f} +/- {std:.4f}" for metric, (mean, std) in results.items()}
    for name, results in model_results.items()
}).T
print(comparison.to_string())

# Bar chart of F1 scores
f1_means = {name: results["f1"][0] for name, results in model_results.items()}
f1_stds = {name: results["f1"][1] for name, results in model_results.items()}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(f1_means.keys())
means = list(f1_means.values())
stds = list(f1_stds.values())
bars = ax.bar(names, means, yerr=stds, capsize=5, color=["#3498db", "#2ecc71", "#e74c3c", "#f39c12"])
ax.set_title("Model Comparison — F1 Score (CV)")
ax.set_ylabel("F1 Score")
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{mean:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

# Select best
best_name = max(f1_means, key=f1_means.get)
print(f"\nBest model: {best_name} (F1 = {f1_means[best_name]:.4f})")

## 10. Final Model

In [ ]:
# Select the best model object
model_objects = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
}
final_model = model_objects[best_name]

# Train on all data
final_model.fit(X, y)
print(f"Final model ({best_name}) trained on {X.shape[0]} samples")

# Sanity check — predict on training data
train_preds = final_model.predict(X)
print(f"Training F1: {f1_score(y, train_preds):.4f}")
print(f"Prediction distribution: {dict(pd.Series(train_preds).value_counts())}")

In [ ]:
# Save model
model_artifact = {
    "model": final_model,
    "model_name": best_name,
    "feature_columns": list(X.columns),
    "target": "binary",
    "cv_f1": f1_means[best_name],
}

model_path = Path("best_model.joblib")
joblib.dump(model_artifact, model_path)
print(f"Model saved to {model_path}")

# Register with MLflow
with mlflow.start_run(run_name=f"final_{best_name.lower().replace(' ', '_')}"):
    mlflow.log_metric("final_cv_f1", f1_means[best_name])
    mlflow.log_param("best_model", best_name)
    mlflow.log_param("n_features", X.shape[1])
    mlflow.log_param("n_samples", X.shape[0])
    mlflow.log_artifact(str(model_path))
    print("Registered with MLflow")